In [ ]:
*** Settings ***
Variables  /home/ute/testline_configuration/7_15_UTE5GTL317
Library    taf.ul.gnb.bip.collection   ${tl}    WITH NAME    bip_collection

In [ ]:
Report Test Result
    Online Reporting to ReP   [1]BipTraceCollectionEvaluation
    ...                       qcFilePath=testsuite/Ulm/resources/Common/ResourceCI/bipcollectionevaluation.qc

In [ ]:
*** Settings ***
Resource    ../ute_workarounds.robot

*** Keywords ***
Bring Airphone to on Air status
    UTE Cloud Workaround
    Bring Airphone TL to on Air status without reboot

In [ ]:
*** Test Cases ***
Test Bip Collection
    Start Bip Trace
    Sleep   10 sec
    ${bip_trace}=   Stop Bip Trace   target_dir=/tmp
    Set Suite Variable   ${bip_trace}

In [ ]:
*** Settings ***
Library   OperatingSystem

*** Test Cases ***
Check file size and such
        ${file_size}    Get file Size   ${bip_trace}


In [ ]:
*** Settings ***
Resource    testsuite/Ulm/resources/Common/python/reporting.py
Resource    testsuite/Ulm/resources/Common/testline.robot

*** Keywords ***
Report Result To Quality Center
    [Arguments]    ${tsResult}
    
    
    ${enbSwVersion}=   get eNB SW Version if eNB is present   ${enb_id}
    ${gnbSwVersion}=   get gNB SW Build Version   ${gnb_id}   ${resultDir}
    ${swBuild}=   construct SW BUILD from   ${gnbSwVersion}   ${enbSwVersion}   ${resultDir}
    ${testLine}=   construct TESTLINE from   ${resultDir}
    
    report Results to ReP   tcName=[1]BipTraceCollectionEvaluation
    ...                     tsResult=${tsResult}
    ...                     swBuild=${swBuild}
    ...                     testLine=${testLine}
    ...                     suitePath=${SUITE SOURCE}
    ...                     qcTestlabPath=/Root/_Training_/RA_Lx_SW_ULM/testautomation/
    ...                     qcTestSetName=BipTraceCollectionEvaluation
    ...                     pmCounters=${EMPTY}
    ...                     featuresCovered=${EMPTY}
    ...                     isForDebugging=Yes

In [ ]:
%%python module BipAnalysis
import os
from robot.libraries.BuiltIn import BuiltIn
rbi = BuiltIn()


class BipAnalysis:

    def get_software_release(self):
        swVersion = rbi.run_keyword("Get gNB SW version")
        if "SBTS" in swVersion:
            return(swVersion)
        else:
            return(swVersion.split("_")[1])


    def download_and_unzip_dissectors(self, swVersion, lua_base_path='/opt/lua_dissector'):

        if not os.path.exists(lua_base_path):
            cmdStr = "sudo mkdir {0} ; sudo chmod 777 {0} ; cd {0}".format(lua_base_path)
            os.system(cmdStr)

        sw_version_path = "{}/{}".format(lua_base_path, swVersion)
        if not os.path.exists(sw_version_path):
            cmdStr = "sudo mkdir {0} ; sudo chmod 777 {0} ; cd {0}".format(sw_version_path)
            os.system(cmdStr)
            os.chdir(sw_version_path)
            if "SBTS" in sw_version_path:
                wget_command = "wget https://lterel.es-si-s3-z4.eecloud.nsn-net.net/build/%s/%s_luashark.zip" %\
                               (swVersion.split("_")[0], swVersion)
            else:
                wget_command = "wget https://artifactory-espoo1.ext.net.nokia.com/artifactory/mnp5g-central-public-local/"
                wget_command += "System_Release/%s/luashark-%s.zip" % (swVersion, swVersion)
            os.system(wget_command)
            os.system("unzip * ; chmod 777 *")
        else:
            print("Dissectors already loaded!")
        luashark_path = sw_version_path + "/generated_luashark/luashark.lua"
        return luashark_path
    
    def get_lua_dissectors_if_needed(self, swVersion=None, lua_base_path='/opt/lua_dissector'):
        # https://artifactory-espoo1.ext.net.nokia.com/artifactory/
        #         mnp5g-central-public-local/System_Release/0.800.2402/luashark-0.800.2402.zip

        try:
            if not swVersion:
                swVersion = get_software_release()
            lua_path = download_and_unzip_dissectors(swVersion, lua_base_path)
            return lua_path
        except Exception as e:
            rbi.log("get_lua_dissectors_if_needed(): Exception = <%s>" % str(e))

In [ ]:
*** Settings ***
Resource   ../testline.robot
Library    BipAnalysis


*** Test Cases ***
Analyse the collected bip trace
    
    ${gnb_sw_version}=    Get gNB SW version
    ${luashark_path}=     download_and_unzip_dissectors   ${gnb_sw_version}
    

In [ ]:
%%python module PcapAnalsis
import os
import subprocess
import csv
import logging
import argparse
import json
from collections import defaultdict

CONFIG = {"tshark":  "/usr/bin/tshark",
          "dissector_path":  '/opt/lua_dissector/SBTS22R2_ENB_0000_000438_000000/generated_luashark/luashark.lua'}

logger = logging.getLogger()


def setup_logging(root_path):
    root_logger = logging.getLogger()
    root_logger.setLevel(logging.DEBUG)
    log_formatter = logging.Formatter("%(asctime)s [%(threadName)-12.12s] [%(levelname)-5.5s] %(message)s")

    file_handler = logging.FileHandler(root_path + '/analyse.log', mode="w")
    file_handler.setFormatter(log_formatter)
    root_logger.addHandler(file_handler)

    console_handler = logging.StreamHandler()
    console_handler.setFormatter(log_formatter)
    root_logger.addHandler(console_handler)
    
    
class Tshark:
    """
    Simple wrapper for command line tshark calls
    """

    def __init__(self, tshark, dissector_path):
        self.tshark = tshark
        self.dissector_path = os.path.normpath(dissector_path)
        self.cmd = [self.tshark, "-X", 'lua_script:{}'.format(dissector_path)]
        self.fields = None

    def exec_cmd(self, cmd):
        """Executes the current setup tshark"""
        ws_dir, _ = os.path.split(self.tshark)
        command_to_execute = self.cmd + cmd
        logger.info('Executing command: {}'.format(" ".join(command_to_execute)))

        stdout, stderr = subprocess.Popen(command_to_execute,
                                          stdout=subprocess.PIPE,
                                          stderr=subprocess.PIPE).communicate()
        data = stdout
        error_data = stderr

        if stderr:
            logger.error(error_data)

        return data, error_data

    def filter(self, input_pcap, display_filter, output_pcap=None):
        logger.info("Filtering file [{}] using display filter [{}]".format(input_pcap, display_filter))
        """Adds a display filter to the tshark call"""
        if not output_pcap:
            pcap_dir, _ = os.path.split(input_pcap)
            output_pcap = os.path.join(pcap_dir, "filtered.pcap")

        cmd = ["-Y", display_filter]
        cmd += ["-r", input_pcap]
        cmd += ["-w", output_pcap]
        self.exec_cmd(cmd)
        return output_pcap

    def extract_fields(self, input_pcap, fields, output_file=None):
        """Adds functionality to extract fields from messages and storing
        it in csv"""
        self.fields = fields

        if not output_file:
            pcap_dir, _ = os.path.split(input_pcap)
            output_file = os.path.join(pcap_dir, "pcap_fields.csv")

        cmd = ["-r", input_pcap]
        cmd += ["-T", "text"]
        cmd += ["-T", "fields"]
        cmd += ["-E", "separator=,"]
        cmd += ["-E", "header=y"]
        cmd += ["-E", "quote=d"]

        for field in fields:
            cmd += ["-e", field]

        data, error_data = self.exec_cmd(cmd)

        # write stdout to file
        with open(output_file, mode='wb') as out_fp:
            out_fp.write(data)

        return output_file

    
    
test_data = {'uldata_pucchreceiverespps':
                 {'sfn':              'UlData_PucchReceiveRespPs.PucchReceiveRespPs.sfn',
                     'slot':             'UlData_PucchReceiveRespPs.PucchReceiveRespPs.slot',
                     'rnti':             'UlData_PucchReceiveRespPs.pucchResources_item_0.rnti',
                     'startPrb':         'UlData_PucchReceiveRespPs.pucchResources_item_0.startPrb',
                     'pucchFormat':      'UlData_PucchReceiveRespPs.pucchResources_item_0.pucchFormat'},
                 'uldata_pucchreceivereq':
                 {'sfn':                     'UlData_PucchReceiveReq.PucchReceiveReq.sfn',
                     'slot':                 'UlData_PucchReceiveReq.PucchReceiveReq.slot',
                     'rnti':                 'UlData_PucchReceiveReq.pucchResources_item_0.rnti',
                     'pucchFormat':          'UlData_PucchReceiveReq.pucchResources_item_0.pucchFormat',
                     'frequencyHopping':     'UlData_PucchReceiveReq.pucchResources_item_0.frequencyHopping',
                     'startPrb':             'UlData_PucchReceiveReq.pucchResources_item_0.startPrb',
                     'numOfPrb':             'UlData_PucchReceiveReq.pucchResources_item_0.numOfPrb',
                     'secondHopPrb':         'UlData_PucchReceiveReq.pucchResources_item_0.secondHopPrb',
                     'numOfSymbols':         'UlData_PucchReceiveReq.pucchResources_item_0.numOfSymbols',
                     'numOfBitsOfUciInformation':
                         'UlData_PucchReceiveReq.pucchResources_item_0.numOfBitsOfUciInformation',
                     'nANPucch': 'UlData_PucchReceiveReq.pucchResources_item_0.nANPucch',
                     'modulationType':   'UlData_PucchReceiveReq.pucchResources_item_0.modulationType'},
            'uldata_prachreceivereq':
                 {'sfn':    'UlData_PrachReceiveReq.PrachReceiveReq.sfn',
                  'slot':   'UlData_PrachReceiveReq.PrachReceiveReq.slot'}}    

class PcapAnalsis():
    def __init__(self):
        self.pcap_file = None
        self.test_data = None
        self.csv_file = None
        self.filtered = None
        self.extracted_fields_csv = None

    def create_analysis_data(self, pcap_file, test_data=test_data):
        self.pcap_file = pcap_file
        self.test_data = test_data
        self.filter_capture_file()
        extracted_csv = self.extract_tshark_fields()
        self.analysis_data = self.translate_tshark_data(extracted_csv)
        self.write_result_file()
        return self.analysis_data
    
    def filter_capture_file(self):
        """
        Filter the pcap file for only containing data relevant to test_data
        """
        tshark_analysis = Tshark(
            CONFIG["tshark"], CONFIG["dissector_path"])

        display_filter = "||".join(self.test_data.keys())
        pcap_base_file_name, _ = os.path.splitext(self.pcap_file)
        self.filtered = pcap_base_file_name + "_filtered.pcap"
        tshark_analysis.filter(self.pcap_file, 
                               display_filter,
                               output_pcap=self.filtered)
        logger.info("Created Filtered pcap file: {}".format(self.filtered))
        return self.filtered

    def extract_tshark_fields(self):
        tshark_analysis = Tshark(
            CONFIG["tshark"], CONFIG["dissector_path"])

        fields_to_collect = list()
        for message, fields in self.test_data.items():
            fields_to_collect += [field_path for _, field_path in fields.items()]

        self.extracted_fields_csv = tshark_analysis.extract_fields(self.filtered, fields_to_collect)
        logger.info("Extracted pcap field data to CSV: {}".format(self.extracted_fields_csv))


    def read_csv_file(self, csv_filename):
        csv_data = list()
        with open(csv_filename) as csvfile:
            data_reader = csv.DictReader(csvfile)
            for row in data_reader:
                csv_data.append(row)
        return csv_data


    def translate_tshark_data(self, extracted_csv_file):
        pcap_csv = self.read_csv_file(self.extracted_fields_csv)
        
        field_dict = dict()
        for message, fields in self.test_data.items():
            for key, field_names in fields.items():
                field_dict[field_names] = (message, key)

        analysis_data = list()
        for csv_item in pcap_csv:
            message = dict()
            msg = ''
            for csv_key, csv_value in [(csv_key, csv_value) for csv_key, csv_value in csv_item.items() if csv_value]:
                msg, key = field_dict[csv_key]
                message[key] = csv_value
            message['message'] = msg
            analysis_data.append(message)
        return analysis_data


    def write_result_file(self, result_file=None):

        if not result_file:
            base_name, _ = os.path.splitext(self.pcap_file)
            result_file = base_name + ".csv"

        csv_field_names = ['message']
        for message_field_description in self.test_data.values():
            for field_name in message_field_description:
                csv_field_names.append(field_name)

        with open(result_file, "w") as csvfile:
            writer = csv.DictWriter(
                csvfile, fieldnames=csv_field_names, extrasaction='ignore')
            writer.writeheader()
            for analysis_item in self.analysis_data:
                writer.writerow(analysis_item)
        logger.info("Filtered Result data written to file {}".format(result_file))
        return result_file

In [ ]:
*** Settings ***
Library   PcapAnalsis

*** Test Cases ***
Test Bip Trace Analysis
        Create Analysis Data    ${bip_trace}

In [ ]:
*** Settings ***
Resource   ../testline.robot

*** Test Cases ***
Get the gNodeB SW Version
    ${sw_version}=    Get gNB SW version
    Set Suite Variable    ${sw_version}